In [58]:
%%capture output
! pip install ibllib
! pip install pynapple
! pip install git+https://github.com/int-brain-lab/paper-brain-wide-map.git
! pip install -U google-colab

In [59]:
# system
from pathlib import Path
from tqdm import tqdm
import pickle

# analysis
import numpy as np
import pandas as pd
import pynapple as nap
from scipy import stats
import glob 
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
import os

import scipy.ndimage
from iblutil.numerical import bincount2D
from iblatlas.atlas import BrainRegions
from brainbox.io.one import SpikeSortingLoader, SessionLoader


# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
import patsy
from scipy.stats import pearsonr
from matplotlib.collections import LineCollection

from umap import UMAP
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from collections import defaultdict

from reproducible_ephys_functions import get_insertions
from functions import idxs_from_files
from one.api import ONE
one= ONE()

In [60]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'


# Load trial data

In [61]:
insertions = get_insertions(level=0, one=one, freeze='freeze_2024_03')
repro_ephys_insertions = pd.DataFrame.from_dict(insertions)

In [62]:
repro_eid = repro_ephys_insertions['probe_insertion']

In [63]:
import brainwidemap
# this dataframe holds a summary of all the sessions
# and for us importantly, the eids and pids
bwm_df = brainwidemap.bwm_query()  # each row of this dataframe is a recording

n_sessions = bwm_df["eid"].unique().shape[0]
n_insertions = bwm_df["pid"].unique().shape[0]
print(
    f"{n_sessions} sessions with {n_insertions} individual neuropixel recordings"
)
# bwm_df.head()
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## Getting units from the brainwide map

In [64]:
units_df = brainwidemap.bwm_units(one)
n_units = units_df.shape[0]
print(f"{n_units} units present in the table")
# units_df.head()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
62990 units present in the table


In [65]:
# cluster_df = pd.read_pickle(data_path+'extended_mouse_LDA_5_bins_cut')
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
# session_cluster = pd.read_parquet(data_path+'cluster_per_session')
lda = pd.read_pickle(data_path+'extended_mouse_LDA_5_bins_cut')
lda = pd.read_pickle(data_path+'mouse_LDA_5_bins_cut18-06-2026')

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

In [66]:
print(len(lda_eid))
print(len(lda_pid))

244
380


In [67]:
use_eid = lda_pid
# use_eid = repro_eid
# use_eid = bwm_pid

In [68]:
len(use_eid)

380

In [69]:
repro_units = units_df.loc[units_df['pid'].isin(list(use_eid))]

## Querying and loading the data

Then we will query the brain region ACA. ACA has many sub-regions, and we need to get the list of neurons belinging to any of the sub-regions.

In [70]:
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']  # Reproducible ephys regions

regions = BrainRegions()
aca_leaf_nodes = regions.descendants(regions.acronym2id(BRAIN_REGIONS))
# print(f"List of regions to query: \n {aca_leaf_nodes['acronym']}")
BRAIN_REGIONS = repro_units['Beryl'].unique()  # All available in the session query

# Find relevant behavioral data

In [71]:
# LOAD DATA
data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

# Identify sessions available to process
sessions_to_process = []
for m, mat in enumerate(idxs):
    mouse_name = mat[37:]
    session = mat[:36]
        
    sessions_to_process.append((mouse_name, session))
print(f"Found {len(sessions_to_process)} sessions to process.")

Found 319 sessions to process.


# Process and save PETH with trial info

In [72]:
states_path = prefix + 'representation_learning_variability/paper-individuality/data/states_files/'
save_states_path = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'

In [85]:
files = os.listdir(save_states_path)
len(files)

0

In [ ]:
total_files = len(list(use_eid))
for p, pid in enumerate(list(use_eid)):
    if (p + 1) % max(1, total_files // 10) == 0 or (p + 1) == total_files:
        print(f"Progress: {(p + 1) / total_files * 100:.1f}% ({p + 1}/{total_files}) files processed")
    check_filename = "states_neurons_file_" + str(pid) 
    if check_filename not in files:
        ssl = SpikeSortingLoader(one=one, pid=pid)
        eid = ssl.eid
        session_files = glob.glob(os.path.join(states_path, f"*{eid}*"))
        # Process ephys sessions if they have behavior
        if session_files:
            session_file_path = session_files[0]  # Take the first (and only) match
            states_trial = pd.read_parquet(session_file_path)
            identifiable_states = states_trial["identifiable_states"] #.dropna()
            digits = identifiable_states.str.extract(r'(\d)(\d)(\d)').astype('Int64')
            digits.columns = ["paw", "whisk", "lick"]
            states_trial['paw'] = digits['paw']
            states_trial['whisk'] = digits['whisk']
            states_trial['lick'] = digits['lick']
            
            # Initialize the final dataframe with states_trial
            states_trial_with_spikes = states_trial.copy()
            # Add session identifier
            states_trial_with_spikes['session_eid'] = eid
            states_trial_with_spikes['session_pid'] = pid
            
            # Assuming states_trial has 'Bin' column with timestamps and some bin width
            bin_width = states_trial_with_spikes['Bin'].diff().median()  # or specify manually
            spikes, clusters, channels = ssl.load_spike_sorting(good_units=True, revision='2025-05-26')
            df_clus = pd.DataFrame(ssl.merge_clusters(spikes, clusters, channels))
            
            # This is where we select the units belonging to each available area
            unique_areas = df_clus['atlas_id'].unique()
            for a, area in enumerate(unique_areas):
                regions = BrainRegions()
                area_name = regions.id2acronym(area)
                aca_leaf_nodes = regions.descendants(regions.acronym2id(area_name))
                # This is where we select the units belonging to any of the leaf nodes brain regions
                selection_clusters = df_clus['atlas_id'].isin(aca_leaf_nodes['id'])
                iclusters = np.where(selection_clusters)[0]
                ispikes = np.isin(spikes['clusters'], iclusters)
                st = spikes['times'][ispikes]
                sc = spikes['clusters'][ispikes]
                    
                # Get the time bins from states_trial
                time_bins = states_trial_with_spikes['Bin'].values
                expected_bin_width = 1/60  

                # Create bin edges: center each timestamp with fixed width around it
                # For histogram, we need edges, so offset by ±width/2
                bin_edges = np.zeros(len(time_bins) + 1)
                bin_edges[0] = time_bins[0] - expected_bin_width/2
                for i in range(len(time_bins) - 1):
                    # Each edge is midpoint between consecutive timestamps
                    bin_edges[i+1] = (time_bins[i] + time_bins[i+1]) / 2
                bin_edges[-1] = time_bins[-1] + expected_bin_width/2

                # Filter spikes to onlavg_spike_countsy include those within the states_trial time range
                time_mask = (st >= bin_edges[0]) & (st <= bin_edges[-1])
                st_filtered = st[time_mask]
                sc_filtered = sc[time_mask]
            
                cluster_ids = np.unique(sc_filtered)
                for cluster_id in cluster_ids:
                    
                    """Process a single cluster and return results"""
                    cluster_spikes = st_filtered[sc_filtered == cluster_id]
                    # Use histogram to bin spikes into the same time bins as states_trial
                    counts, _ = np.histogram(cluster_spikes, bins=bin_edges)
                    states_trial_with_spikes[f'{area_name[0]}_neuron_{int(cluster_id)}_spike_count'] = counts
                
                states_trial_with_spikes = states_trial_with_spikes.dropna(subset=['trial_id'])
                
                # # Create bin edges for histogram
                # # We need n+1 edges for n bins, so we need to extrapolate the bin edges
                # bin_width = np.median(np.diff(time_bins))  # Estimate bin width (~1/60 = 0.0167 seconds)
                # # Create bin edges: start from half bin width before first bin, end half bin width after last bin
                # bin_edges = np.concatenate([
                #     [time_bins[0] - bin_width/2],  # Start edge
                #     time_bins[:-1] + bin_width/2,  # Middle edges
                #     [time_bins[-1] + bin_width/2]  # End edge
                # ])
                # # Filter spikes to onlavg_spike_countsy include those within the states_trial time range
                # time_mask = (st >= bin_edges[0]) & (st <= bin_edges[-1])
                # st_filtered = st[time_mask]
                # sc_filtered = sc[time_mask]
            
                # cluster_ids = np.unique(sc_filtered)
                # for cluster_id in cluster_ids:
                    
                #     """Process a single cluster and return results"""
                #     cluster_spikes = st_filtered[sc_filtered == cluster_id]
                #     # Use histogram to bin spikes into the same time bins as states_trial
                #     counts, _ = np.histogram(cluster_spikes, bins=bin_edges)
                #     states_trial_with_spikes[f'{area_name[0]}_neuron_{int(cluster_id)}_spike_count'] = counts
                    
            # Save per session
            save_filename = save_states_path + "states_neurons_file_" + str(pid) 
            with open(save_filename.replace('.parquet', '.pkl'), 'wb') as f:
                pickle.dump(states_trial_with_spikes, f)